# Dasar Tata Bahasa (Grammar) & Recursive Descent Parser
Tutorial ini membahas dasar-dasar tata bahasa dalam ilmu komputer, notasi EBNF, dan implementasinya menggunakan teknik *Recursive Descent Parser* di Python.

## 1. Pengertian Dasar Tata Bahasa (Grammar)
Tata bahasa mendefinisikan aturan pembentukan struktur bahasa yang valid.
- **Terminal Symbol**: Unit terkecil (token) seperti `10`, `+`, `if`.
- **Nonterminal Symbol**: Kategori struktur seperti `ekspresi` atau `kalimat`.
- **Aturan Produksi**: Aturan yang menjelaskan bagaimana nonterminal dibentuk.

## 2. Context-Free Grammar (CFG) & EBNF
EBNF (*Extended Backus-Naur Form*) digunakan untuk menuliskan aturan CFG agar lebih ringkas.

| Elemen EBNF | Sintaks | Pemetaan ke Kode Python |
| :--- | :--- | :--- |
| **Terminal** | `"a"` | Memanggil `expect()` |
| **Nonterminal** | `a` | Memanggil fungsi parser terkait |
| **Alternatif** | `a | b` | Logika `if / elif / else` |
| **Repetisi** | `{ a }` | Perulangan `while` |

## 3. Contoh Kasus: Ekspresi Matematika
Aturan tata bahasa untuk operasi matematika dengan prioritas operator:
1. `additive = multiplicative , { ("+" | "-") , multiplicative } ;` 
2. `multiplicative = primary , { ("*" | "/" | "%") , primary } ;` 
3. `primary = integer | "(" , additive , ")" ;` 

In [9]:
import re

class ParserError(Exception):
    pass

class ExpressionParser:
    def __init__(self, source):
        # 1. Updated Regex: \d+\.\d+ matches decimals (e.g., 2.6), \d+ matches integers.
        # The order is important; it checks for decimals first.
        tokens_list = re.findall(r'\d+\.\d+|\d+|[a-zA-Z]+|[+\-*/%()]', source)
        self._tokens = iter(tokens_list + ['?'])
        self._current = None
        self.advance()

    def advance(self):
        """Move to the next token in the stream."""
        try:
            self._current = next(self._tokens)
        except StopIteration:
            self._current = None

    def expect(self, expected_list):
        """Verify the current token matches expectations before consuming it."""
        if self._current not in expected_list:
            raise ParserError(f"Expected {expected_list}, found {self._current}")
        token = self._current
        self.advance()
        return token

    def primary(self):
        """
        primary = float | "(" , additive , ")"
        """
        token = self._current
        
        # 2. Support for floats: Check if the token is a number (contains digits or a dot)
        if token and (token.replace('.', '', 1).isdigit()):
            # Convert to float to maintain decimal precision
            return float(self.expect([token]))
            
        elif token == '(':
            self.advance()
            result = self.additive()
            self.expect([')'])
            return result
        else:
            raise ParserError(f"Unexpected token: {token}")

    def multiplicative(self):
        """
        multiplicative = primary , { ("*" | "/" | "%") , primary }
        """
        result = self.primary()
        while self._current in ('*', '/', '%'):
            op = self.expect(['*', '/', '%'])
            right = self.primary()
            if op == '*': 
                result *= right
            elif op == '/': 
                # 3. Use true division (/) instead of floor division (//) for floats
                result /= right 
            elif op == '%': 
                result %= right
        return result

    def additive(self):
        """
        additive = multiplicative , { ("+" | "-") , multiplicative }
        """
        result = self.multiplicative()
        while self._current in ('+', '-'):
            op = self.expect(['+', '-'])
            right = self.multiplicative()
            if op == '+': 
                result += right
            elif op == '-': 
                result -= right
        return result

# Simulation
expr = "1 + 10 * (2 * (1.5 - 1))"
parser = ExpressionParser(expr)
print(f"Ekspresi: {expr}")
print(f"Hasil Evaluasi: {parser.additive()}") # Output should be 15.2

Ekspresi: 1 + 10 * (2 * (1.5 - 1))
Hasil Evaluasi: 11.0


Bayangkan aturan tata bahasa (EBNF) ini seperti struktur organisasi di sebuah pabrik matematika. Ada 3 jabatan dari atas ke bawah:

### additive (Si Bos Besar):
Tugasnya hanya mengurus penjumlahan (+) dan pengurangan (-). Tapi, sebelum dia menjumlahkan sesuatu, dia akan selalu menyuruh bawahannya (multiplicative) untuk bekerja duluan.

### multiplicative (Si Manajer):
Tugasnya mengurus perkalian (*), pembagian (/), dan sisa bagi (%). Sama seperti bosnya, sebelum dia mengalikan sesuatu, dia butuh bahan baku dari bawahannya (primary).

### primary (Si Staf Lapangan):
Ini adalah ujung tombak. Tugasnya sangat sederhana: mencari angka dasar (seperti 10, 2.6) ATAU mengurus apapun yang ada di dalam tanda kurung ( ... ).

<B>Kenapa strukturnya dibuat begini?</B></Br>
Karena dalam matematika, perkalian/pembagian harus dikerjakan lebih dulu daripada penjumlahan/pengurangan. Dengan membuat additive bergantung pada multiplicative, kita memaksa komputer untuk menyelesaikan perkalian terlebih dahulu.

## 4. Penjelasan Mekanisme Parser
- **Prioritas Operator**: Fungsi `additive` memanggil `multiplicative`, sehingga perkalian dikerjakan lebih dulu (hierarki fungsi).
- **Asosiativitas Kiri**: Penggunaan `while` (bukan rekursi langsung) memastikan evaluasi berjalan dari kiri ke kanan.
- **Penanganan Error**: `ParserError` menangkap input yang tidak valid sesuai aturan EBNF.

Mekanisme Cara Kerjanya (Praktek di Lapangan)

Mari kita lihat bagaimana struktur organisasi di atas bekerja di dunia nyata saat memproses soal: 10 + 2 * 3

### 1. Prioritas Operator (Siapa kerja duluan?)

Program mulai dari Si Bos Besar (additive).

Bos melihat ada soal. Dia langsung menyuruh Manajer (multiplicative) mencari angka pertama. Manajer menyuruh Staf (primary). Staf mengambil angka 10.

Bos melihat tanda +. Sebelum dia bisa menjumlahkan 10 dengan angka selanjutnya, dia menyuruh Manajer lagi untuk mencari angka berikutnya.

Manajer melihat angka 2. Tapi tunggu, setelah angka 2, ada tanda * (perkalian)! Karena ini tugas Manajer, Manajer langsung mengambil angka 3 dan mengalikannya. 2 * 3 = 6.

Manajer menyerahkan hasil 6 ke Bos.

Bos menjumlahkannya: 10 + 6 = 16.

Intinya: Karena kode dipecah menjadi fungsi-fungsi yang saling memanggil, perkalian (2 * 3) otomatis dieksekusi sebelum penjumlahan (+) menyelesaikannya tugasnya.

### 2. Asosiativitas Kiri (Membaca dari Kiri ke Kanan)
Bagaimana jika soalnya: 10 - 5 - 2?

Dalam matematika, ini harus dihitung dari kiri: (10 - 5) = 5, lalu 5 - 2 = 3.

Jika kita salah membuat aturan, komputer bisa saja menghitung yang kanan duluan: 10 - (5 - 2) = 7 (Ini salah!).

Untuk mencegah ini, kita menggunakan perulangan while di dalam fungsi Python-nya. Perulangan while memaksa komputer untuk terus "memakan" operasi dari kiri ke kanan satu per satu sampai habis.

### 3. Penanganan Error (Satpam Sintaks)

Fungsi expect() dan ParserError bertindak seperti satpam.

Jika Anda memasukkan input yang aneh, misalnya 10 + * 3, Bos (additive) melihat +, lalu meminta Manajer mengambil angka. Tapi Manajer malah melihat *. Karena * bukan angka, satpam akan langsung membunyikan alarm (Error): "Hei, saya mengharapkan angka, tapi malah menemukan tanda bintang!"